# 06 Imaging Cascade Pipeline

Run cascade-based spike inference and inspect outputs.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:
import os
import sys

from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')

import datajoint as dj
from adamacs.pipeline import subject, session, equipment, surgery, event, trial, imaging, behavior, scan, model, analysis, denoising
from adamacs.ingest import session as isess
from adamacs.ingest import behavior as ibe
from adamacs.utility import *
from adamacs.helpers import stack_helpers as sh
import re
import numpy as np

dj.__version__

print(dj.__version__)
print(dj.config['custom']['database.prefix'])


def find_max_iteration_file(directory):
    files = [
        f for f in os.listdir(directory)
        if (match := re.match(r"model_(\d+)\.pth", f))
    ]
    return os.path.join(directory, max(files, key=lambda x: int(re.search(r"model_(\d+)\.pth", x).group(1)))) if files else None


# search for specific models

In [ ]:
imaging.ActivityCascadeModel & f"model_name LIKE '%Global%30%high%'" # look for the model name in the database


In [ ]:
imaging.ActivityCascadeModel & f"model_name LIKE '%8s%'" # look for the model name in the database

# Generate and insert Cascade tasks

example single scan_id

In [ ]:
scan_id = "scan9FS204TD"
scan_key = (scan.Scan & f'scan_id = "{scan_id}"').fetch1('KEY')
imaging.Curation & scan_key

In [ ]:
scan_id = "scan9FS204TD"

paramsetidx = 10
curation = 5

scan_key = (scan.Scan & f'scan_id = "{scan_id}"').fetch1('KEY')

indicator = (subject.Subject * session.Session()  * subject.Line()  & scan_key).fetch1('line_name')
print(f"Indicator: {indicator}")

if 'GCaMP8s' in indicator:
    modelname = 'GC8s_EXC_30Hz_smoothing50ms_high_noise'
elif 'GCaMP6s' in indicator:
    modelname = 'Global_EXC_30Hz_smoothing50ms_high_noise'

print(f"Model name: {modelname}")

insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
    & scan_key
    & f'paramset_idx = {paramsetidx}'
    & f'curation_id = {curation}'
    & 'extraction_method = "cascade_inference"').fetch1()

insertkey['model_name'] = modelname

imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)


example bulk processing over range of scans

In [ ]:
# # example: all scans from AM belonging to a specific samesite_id
# scans_to_process = (session.Session * session.SessionUser * subject.User * event.BehaviorRecording & "session_datetime >= '2024-12-02'" & "initials = 'NK'").fetch("KEY")

# # example: all scans from AM after a certain date, with a curation later than a specific date
scans_to_process = (session.Session * session.SessionUser * subject.User * event.BehaviorRecording & "session_datetime >= '2024-01-02'" & "initials = 'LE'").fetch("KEY")
# scans_to_process = (imaging.Curation & scans_to_process & "curation_time >= '2024-09-01'").fetch("KEY")

# # example: all scans from LK belonging to a specific samesite_id
# samesite_id = 'sess9FS1DQX9'
# samesite_session_key = (session.Session * session.SessionSameSite * session.SessionNote  & f'same_site_id = "{samesite_id}"' & 'session_note = "Natural Images"').fetch('KEY')
# scans_to_process = (scan.Scan & samesite_session_key).fetch('KEY')

scans_to_process


In [ ]:
session.Session * session.SessionSameSite * scan.Scan & 'scan_id = "scan9FS204TD"'

In [ ]:
# Define samesite_id for filtering scans
# samesite_id = 'sess9FS1DQX9'

paramsetidx = 10
# curation = 1
# # Fetch the session key for the given samesite_id
# samesite_session_key = (session.Session * session.SessionSameSite * imaging.Curation & f'same_site_id = "{samesite_id}"' & f'curation_id ="{curation}"').fetch('KEY')

# # Fetch all the scans to process
# scans_to_process = (scan.Scan & samesite_session_key).fetch('KEY')

# paramsetidx = 10
# curation = 3

# Loop over all scans to process
for scan_key in scans_to_process:
    try:
        # Fetch the indicator (line_name) for the current scan
        indicator = (subject.Subject * session.Session() * subject.Line() & scan_key).fetch1('line_name')
        print(f"Indicator: {indicator}")

        # Define the model name based on the indicator
        if 'GCaMP8s' in indicator:
            modelname = 'GC8s_EXC_30Hz_smoothing50ms_high_noise'
        elif 'GCaMP6s' in indicator:
            modelname = 'Global_EXC_30Hz_smoothing50ms_high_noise'
        else:
            modelname = 'Unknown_Model'  # Handle unexpected cases if necessary

        print(f"Model name: {modelname}")
        
        latest_curation = (imaging.Curation & scan_key).fetch("curation_id").max()
        
        # Fetch the insertkey for ActivityCascadeTask
        insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
            & scan_key
            & f'paramset_idx = {paramsetidx}'
            & f'curation_id = {latest_curation}'
            & 'extraction_method = "cascade_inference_11"').fetch1()

        # Add the model name to the insertkey
        insertkey['model_name'] = modelname

        # Insert into ActivityCascadeTask, ensuring we ignore extra fields and skip duplicates
        imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)
    except Exception as e:
        print(f"Error processing scan {scan_key}: {e}")

In [ ]:
(imaging.ActivityCascadeTask() * session.SessionSameSite & scans_to_process).fetch(as_dict=True)

In [ ]:
# Fetch all the scans to process REDO ALL
scans_to_process = (imaging.ActivityCascadeTask & 'extraction_method = "cascade_inference"').fetch('KEY')

# Loop over all scans to process
for scan_key in scans_to_process:
    # Fetch the indicator (line_name) for the current scan
    indicator = (subject.Subject * session.Session() * subject.Line() & scan_key).fetch1('line_name')
    print(f"Indicator: {indicator}")

    # Define the model name based on the indicator
    if 'GCaMP8s' in indicator:
        modelname = 'GC8s_EXC_30Hz_smoothing25ms_high_noise'
    elif 'GCaMP6s' in indicator:
        modelname = 'Global_EXC_30Hz_smoothing50ms_high_noise'
    else:
        modelname = 'Unknown_Model'  # Handle unexpected cases if necessary

    print(f"Model name: {modelname}")

    paramsetidx = scan_key['paramset_idx']
    curation = scan_key['curation_id']
    
    
    # Fetch the insertkey for ActivityCascadeTask
    insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
        & scan_key
        & f'paramset_idx = {paramsetidx}'
        & f'curation_id = {curation}'
        & 'extraction_method = "cascade_inference"').fetch1()

    # Add the model name to the insertkey
    insertkey['model_name'] = modelname

    # Insert into ActivityCascadeTask, ensuring we ignore extra fields and skip duplicates
    imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)

In [ ]:
(jobtable & 'status="error"').delete()

In [ ]:
(jobtable & 'status="error"').delete()

In [ ]:
jobtable

In [ ]:
imaging.ActivityExtractionMethod()

In [ ]:
# Fetch all the scans to process REDO ALL WITHOUT NEUROPIL
scans_to_process = (imaging.ActivityCascadeTask & 'extraction_method = "cascade_inference"').fetch('KEY')

# Loop over all scans to process
for scan_key in scans_to_process:
    # Fetch the indicator (line_name) for the current scan
    indicator = (subject.Subject * session.Session() * subject.Line() & scan_key).fetch1('line_name')
    print(f"Indicator: {indicator}")

    # Define the model name based on the indicator
    if 'GCaMP8s' in indicator:
        modelname = 'GC8s_EXC_30Hz_smoothing25ms_high_noise'
    elif 'GCaMP6s' in indicator:
        modelname = 'Global_EXC_30Hz_smoothing50ms_high_noise'
    else:
        modelname = 'Unknown_Model'  # Handle unexpected cases if necessary

    print(f"Model name: {modelname}")

    paramsetidx = scan_key['paramset_idx']
    curation = scan_key['curation_id']
    
    
    # Fetch the insertkey for ActivityCascadeTask
    insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
        & scan_key
        & f'paramset_idx = {paramsetidx}'
        & f'curation_id = {curation}'
        & 'extraction_method = "cascade_inference"').fetch1()

    # Add the model name to the insertkey
    insertkey['model_name'] = modelname
    insertkey['extraction_method'] = 'cascade_inference_11'
   
    # Insert into ActivityCascadeTask, ensuring we ignore extra fields and skip duplicates
    imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)

In [ ]:
imaging.ProcessingParamSet()

In [ ]:
paramsetidx

In [ ]:
(imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') 
                              * imaging.ActivityCascadeTask)

In [ ]:
 imaging.ActivityCascadeTask & dj.Not(imaging.Activity())

In [ ]:
(imaging.Activity & 'extraction_method LIKE "%_11%"')

# Populate
SHOULD RUN IN BACKGROUND AUTOMATICALLLY - usually dont do this manually - 

In [ ]:
populate_settings = {'display_progress': True, 'suppress_errors': False, 'reserve_jobs': True}
imaging.Activity.populate(**populate_settings)

# Playground to test the code

plot some traces

In [ ]:
from tqdm import tqdm
from scipy.ndimage import percentile_filter
from joblib import Parallel, delayed
from pathlib import Path
import os
import sys

cascade_candidates = []
env_cascade = os.environ.get('CASCADE_PATH')
if env_cascade:
    cascade_candidates.append(Path(env_cascade))
cascade_candidates.extend([
    Path.home() / 'github' / 'Cascade',
    Path.home() / 'Cascade',
])

cascade = None
for cascade_path in cascade_candidates:
    if cascade_path.is_dir():
        sys.path.append(str(cascade_path))
        os.chdir(cascade_path)
        from cascade2p import cascade
        print(f"Using Cascade at {cascade_path}")
        break

if cascade is None:
    raise FileNotFoundError(
        'Could not find Cascade. Set CASCADE_PATH or place Cascade under ~/github/Cascade.'
    )

In [ ]:
scan_id = "sess9FS204TD"
paramsetidx = 10
curation = 6

scan_key = (imaging.ActivityCascadeTask  & imaging.Activity.Trace & f'scan_id = "{scan_id}"' & f'paramset_idx = "{paramsetidx}"' & f'curation_id = "{curation}"').fetch('KEY')
scan_key

In [ ]:
scan_key = (imaging.ActivityCascadeTask & scan_key & 'extraction_method = "cascade_inference_no_neuropil"').fetch1('KEY')
title = scan_key["extraction_method"]
# scan_key = (imaging.ActivityCascadeTask & scan_key & 'extraction_method = "cascade_inference"').fetch1('KEY')

In [ ]:
imaging.Curation & scan_key

In [ ]:
k = (imaging.Activity.Trace & scan_key).fetch('activity_trace')
k = np.vstack(k)

In [ ]:
f = (imaging.Fluorescence.Trace & scan_key).fetch('fluorescence')
f = np.vstack(f)
normalized_f = (f - np.min(f, axis=1, keepdims=True)) / (np.max(f, axis=1, keepdims=True) - np.min(f, axis=1, keepdims=True)) * 2

In [ ]:
from cascade2p.utils import plot_dFF_traces, plot_noise_level_distribution, plot_noise_matched_ground_truth
import matplotlib.pyplot as plt
framerate = (scan.ScanInfo & scan_key).fetch1('fps')

plt.rcParams['figure.figsize'] = [13, 13]

nb_neurons = 5
neuron_indices = np.random.randint(f.shape[0], size=nb_neurons)
# time_axis = plot_dFF_traces(dFF,neuron_indices,framerate, spikes,y_range=(-1.5, 2))
time_axis = plot_dFF_traces(normalized_f,neuron_indices,framerate, k, y_range=(-1.5, 2))

plt.title(f"{scan_key['scan_id']} {title} {scan_key['paramset_idx']} {scan_key['curation_id']}")

In [ ]:
from cascade2p.utils_discrete_spikes import infer_discrete_spikes 
model_name, model_path = (imaging.ActivityCascadeModel * imaging.ActivityCascadeTask & scan_key).fetch1('model_name', 'model_path')
discrete_approximation, spike_time_estimates  = infer_discrete_spikes(k, model_name, verbosity=1)

In [ ]:
plt.rcParams['figure.figsize'] = [13, 13]

nb_neurons = 32
# neuron_indices = np.random.randint(f.shape[0], size=nb_neurons)
# time_axis = plot_dFF_traces(dFF,neuron_indices,framerate, spikes,y_range=(-1.5, 2))
time_axis = plot_dFF_traces(normalized_f,neuron_indices,framerate, k, y_range=(-1.5, 2))

plt.title(f"{scan_key['scan_id']} {title} {scan_key['paramset_idx']} {scan_key['curation_id']}")

time_axis = plot_dFF_traces(normalized_f,neuron_indices,frame_rate,spiking=k,discrete_spikes=spike_time_estimates )


In [ ]:
from cascade2p.utils_discrete_spikes_parallel_fast import infer_discrete_spikes 
model_name, model_path = (imaging.ActivityCascadeModel * imaging.ActivityCascadeTask & scan_key).fetch1('model_name', 'model_path')
discrete_approximation, spike_time_estimates  = infer_discrete_spikes(k, model_name, verbosity=1)

In [ ]:
plt.rcParams['figure.figsize'] = [13, 13]

nb_neurons = 32
# neuron_indices = np.random.randint(f.shape[0], size=nb_neurons)
# time_axis = plot_dFF_traces(dFF,neuron_indices,framerate, spikes,y_range=(-1.5, 2))
time_axis = plot_dFF_traces(normalized_f,neuron_indices,framerate, k, y_range=(-1.5, 2))

plt.title(f"{scan_key['scan_id']} {title} {scan_key['paramset_idx']} {scan_key['curation_id']}")

time_axis = plot_dFF_traces(normalized_f,neuron_indices,frame_rate,spiking=k,discrete_spikes=spike_time_estimates )


## Cascade loading and model updates

In [ ]:
from tqdm import tqdm
from scipy.ndimage import percentile_filter
from joblib import Parallel, delayed
from pathlib import Path
import os
import sys

cascade_candidates = []
env_cascade = os.environ.get('CASCADE_PATH')
if env_cascade:
    cascade_candidates.append(Path(env_cascade))
cascade_candidates.extend([
    Path.home() / 'github' / 'Cascade',
    Path.home() / 'Cascade',
])

cascade = None
for cascade_path in cascade_candidates:
    if cascade_path.is_dir():
        sys.path.append(str(cascade_path))
        os.chdir(cascade_path)
        from cascade2p import cascade
        print(f"Using Cascade at {cascade_path}")
        break

if cascade is None:
    raise FileNotFoundError(
        'Could not find Cascade. Set CASCADE_PATH or place Cascade under ~/github/Cascade.'
    )

### Cascade models

In [ ]:
# Get list of the names of available models and download them

import ruamel.yaml as yaml
yaml = yaml.YAML(typ='rt')

cascade.download_model( 'update_models',verbose = 1)

yaml_file = open('Pretrained_models/available_models.yaml')
X = yaml.load(yaml_file)
list_of_models = list(X.keys())
print('\n List of available models: \n')
for model in list_of_models:
  print(model)
  # uncomment the next line to download all models
  # cascade.download_model(model, verbose=1)


## insert all models into the database if needed

In [ ]:
# Insert all models into the database
from pathlib import Path
import os

modelfolder = Path(os.environ.get('CASCADE_MODEL_DIR', str(Path.home() / 'Cascade' / 'Pretrained_models')))
for model_name in list_of_models:
    model_path = str(modelfolder / f"{model_name}.pth")
    model_description = f"Pretrained model {model_name}"

    imaging.ActivityCascadeModel.insert1({
        'model_name': model_name,
        'model_path': model_path,
        'model_description': model_description
    }, skip_duplicates=True)

In [ ]:
 imaging.ActivityCascadeModel()

## Older code snippets

Manual Cascade

In [ ]:
scan_id = "scan9FXN7UFU"

paramsetidx = 10
curation = 6
modelname = 'GC8s_EXC_30Hz_smoothing50ms_high_noise'

scan_key = (scan.Scan & f'scan_id = "{scan_id}"').fetch1('KEY')

insertkey = (imaging.Fluorescence * imaging.ProcessingParamSet.proj('processing_method') * imaging.ActivityExtractionMethod
    & scan_key
    & f'paramset_idx = {paramsetidx}'
    & f'curation_id = {curation}'
    & 'extraction_method = "cascade_inference"').fetch1()

insertkey['model_name'] = modelname

# imaging.ActivityCascadeTask.insert1(insertkey, ignore_extra_fields=True, skip_duplicates=True)


In [ ]:
# Retrieve the key and update the model name to the desired one

key = insertkey 
key

In [ ]:
(imaging.ActivityCascadeModel & {'model_name': key['model_name']}).fetch1('model_name', 'model_path')

In [ ]:
# # Load the method and imaging dataset from the curation
# method, imaging_dataset = imaging.get_loader_result(key, imaging.Curation)
# https://github.com/HelmchenLabSoftware/Cascade/blob/master/Demo%20scripts/Process_output_from_Suite2p.py

# Fetch the fluorescence traces from the database, ordered by mask
traces = (imaging.Fluorescence.Trace & key).fetch(as_dict=True, order_by="mask")

# Fetch the model parameters based on the model name
model_name, model_path = (imaging.ActivityCascadeModel & {'model_name': key['model_name']}).fetch1('model_name', 'model_path')

# Fetch the framerate (frames per second) for the scan
framerate = (scan.ScanInfo & key).fetch1('fps')

smoothing_window = framerate * 60
neu = 0.7

# Stack the fluorescence and neuropil fluorescence traces for all timepoints
Fall = np.vstack([trace['fluorescence'] for trace in traces])
Fneu_all = np.vstack([trace['neuropil_fluorescence'] for trace in traces])

# DO DARKFRAME CORRECTION
mean_darksignal = sh.calculate_mean_darksignal(event, Fall, key)
Fall = Fall - mean_darksignal
Fneu_all = Fneu_all - mean_darksignal

# Detrend by subtracting a scaled version of the neuropil fluorescence from the fluorescence signal and adding the median neuropil fluorescence in order to avoid negative values
dF = (Fall - neu * Fneu_all) + (np.nanmedian(Fneu_all, axis=1, keepdims=True) * neu)



In [ ]:
(imaging.Activity & 'extraction_method = "cascade_inference"')

In [ ]:

# Function to compute the baseline (F0) for each trace using percentile filtering
def compute_F0(trace_dF, smoothing_window, percentile=11):
    return percentile_filter(trace_dF, percentile, size=int(smoothing_window))

# os.environ["CUDA_VISIBLE_DEVICES"] = "3" 

# size the F0 computation across all traces using multiple jobs
F0 = np.array(Parallel(n_jobs=-1)(
    delayed(compute_F0)(dF[i, :], smoothing_window) for i in tqdm(range(dF.shape[0]), desc="Calculating F0", ncols=200)
))

# Calculate ΔF/F0 for all traces (normalized fluorescence change)
dFF = (dF - F0) / F0

# Perform spike inference using the cascade model on the ΔF/F0 array
spikes = cascade.predict(model_name, dFF[:, :], verbosity=1)

# Initialize a list to store the inferred spikes for each trace
inferred_spikes = []

# # For each trace, append the inferred spikes along with other trace information
# for trace in tqdm(traces, desc="Inferring spikes"):
#     inferred_spikes.append({
#         **keyn,  # Include the key information
#         'mask': trace['mask'],  # Include the mask for the trace
#         'fluo_channel': trace['fluo_channel'],  # Include the fluorescence channel information
#         'activity_trace': spikes  # Store the inferred activity trace (spikes)
#     })

In [ ]:
ll = np.nanmedian(Fneu_all, axis=1, keepdims=True) * 0.7

In [ ]:
(Fall - 0.7 * Fneu_all) + np.nanmedian(Fneu_all, axis=1, keepdims=True) * 0.7

In [ ]:
smoothing_window

In [ ]:

import matplotlib.pyplot as plt
import random

n = random.randint(0, Fall.shape[0])  # Randomly select a trace index

# Plot F, dF, and F0 for the first trace as an example
plt.figure(figsize=(12, 5))
# plt.plot((Fall[n,:]), label='Mean F (Fluorescence)')
# plt.plot((Fneu_all[n,:]), label='Mean neuropil F (Fluorescence)')
# plt.plot((dF[n,:]), label='Mean dF (F - 0.7 * Fneu) + (0.7 * median(Fneu))')
# plt.plot((F0[n,:]), label='Mean F0 (Baseline)', linestyle='--')
plt.plot((dFF[n,:]), label='Mean F0 (Baseline)', linestyle='-')
plt.xlabel('Frame')
plt.ylabel('Signal')
plt.title('Mean F, dF, and F0 across all traces')
plt.legend()
plt.tight_layout()
plt.xlim(0, 1000)  # Adjust x-axis limits to show the first 1000 frames
plt.show()

In [ ]:
def calculate_mean_darksignal(traces, scan_key, shuttertime=0.09):
    """
    Calculate the mean darksignal of the mean of all traces for a given scan and shuttertime.
    Args:
        traces: list or array of fluorescence traces (each trace is 1D array or dict with 'fluorescence' key)
        scan_key: DataJoint key for the scan
        shuttertime: float, shutter time offset (default: 0.09)
    Returns:
        mean_darksignal: float, mean darksignal value
    """
   
    # Get darkframe times
    darkframe_shutter_start = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_start_time')
    darkframe_shutter_end = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_end_time')
    darkframetimes = [float(darkframe_shutter_end[0]) + shuttertime, float(darkframe_shutter_start[1])  - shuttertime]
    twoptimestamps = (event.Event()  &  'event_type LIKE "%2p_frames%"' &  scan_key ).fetch('event_start_time')
    darkframes = sh.get_closest_timestamps(darkframetimes, float(twoptimestamps))

    print(darkframes)
    
    # Stack traces if needed
    if isinstance(traces[0], dict) and 'fluorescence' in traces[0]:
        traces_stack = np.vstack([tr['fluorescence'] for tr in traces])
    else:
        traces_stack = np.vstack(traces)
    average_trace = np.mean(traces_stack, axis=0)
    mean_darksignal = np.mean(average_trace[darkframes[0]:darkframes[-1]])
    print(f"Mean darksignal: {mean_darksignal}")
    return mean_darksignal

# Example usage:
# mean_darksignal = calculate_mean_darksignal(traces, scan_key)

In [ ]:
# DO DARKFRAME CORRECTION
mean_darksignal = sh.calculate_mean_darksignal(event, Fall, key)
Fall = Fall - mean_darksignal
Fneu_all = Fneu_all - mean_darksignal


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(spikes.flatten()[~np.isnan(spikes.flatten())], bins=50)
plt.xlabel('Spike Value')
plt.ylabel('Frequency')
plt.title('Histogram of Inferred Spikes')
plt.show()

In [ ]:
from cascade2p.utils import plot_dFF_traces, plot_noise_level_distribution, plot_noise_matched_ground_truth
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = [12 , 5]
noise_levels = plot_noise_level_distribution(dFF,framerate)

In [ ]:
plt.rcParams['figure.figsize'] = [13, 13]

nb_neurons = 16
neuron_indices = np.random.randint(dFF[:50, :].shape[0], size=nb_neurons)
time_axis = plot_dFF_traces(dFF[:50, :],neuron_indices,framerate, spikes,y_range=(-1.5, 3))


In [ ]:

# Initialize a list to store the inferred spikes for each trace
inferred_spikes = []

# For each trace, append the inferred spikes along with other trace information
for trace in tqdm(traces, desc="Inferring spikes"):
    inferred_spikes.append({
        **keyn,  # Include the key information
        'mask': trace['mask'],  # Include the mask for the trace
        'fluo_channel': trace['fluo_channel'],  # Include the fluorescence channel information
        'activity_trace': spikes  # Store the inferred activity trace (spikes)
    })

In [ ]:
imaging.Activity.insert1(key)
imaging.Activity.Trace.insert(inferred_spikes)

In [ ]:
(event.Event() & "event_type='shutter'" & key).fetch('event_start_time')

In [ ]:
# Use existing variables; convert Decimal to float to avoid type errors
darkframetimes = [float(darkframe_shutter_end[0]) + shuttertime,
                  float(darkframe_shutter_start[1]) - shuttertime]
darkframetimes

In [ ]:
traces = (imaging.Fluorescence.Trace & key).fetch(as_dict=True, order_by="mask", limit=1)
mean_darksignal = calculate_mean_darksignal(traces, key)

In [ ]:
(imaging.Fluorescence.Trace & key).fetch(as_dict=True, order_by="mask", limit=20)

In [ ]:
# # Load the method and imaging dataset from the curation
# method, imaging_dataset = imaging.get_loader_result(key, imaging.Curation)
# https://github.com/HelmchenLabSoftware/Cascade/blob/master/Demo%20scripts/Process_output_from_Suite2p.py

# Fetch the fluorescence traces from the database, ordered by mask
traces = (imaging.Fluorescence.Trace & key).fetch(as_dict=True, order_by="mask", limit=1)

# Fetch the model parameters based on the model name
model_name, model_path = (imaging.ActivityCascadeModel & {'model_name': key['model_name']}).fetch1('model_name', 'model_path')

# Fetch the framerate (frames per second) for the scan
framerate = (scan.ScanInfo & key).fetch1('fps')

# Calculate the smoothing window size (in samples), assuming it’s 60 seconds of data
framerate = (scan.ScanInfo & key).fetch1('fps')
smoothing_window = framerate * 60

# Stack the fluorescence and neuropil fluorescence traces for all timepoints
Fall = np.vstack([trace['fluorescence'] for trace in traces])
Fneu_all = np.vstack([trace['neuropil_fluorescence'] for trace in traces])

# Detrend by subtracting a scaled version of the neuropil fluorescence from the fluorescence signal
dF = Fall - 0.15 * Fneu_all

# Function to compute the baseline (F0) for each trace using percentile filtering
def compute_F0(trace_dF, smoothing_window):
    return percentile_filter(trace_dF, 15, size=int(smoothing_window))

# Parallelize the F0 computation across all traces using multiple jobs
F0 = np.array(Parallel(n_jobs=-1)(
    delayed(compute_F0)(dF[i, :], smoothing_window) for i in tqdm(range(dF.shape[0]), desc="Calculating F0", ncols=100)
))

# Calculate ΔF/F0 for all traces (normalized fluorescence change)
dFF = (dF - F0) / F0

# Perform spike inference using the cascade model on the ΔF/F0 array
spikes = cascade.predict(model_name, dFF[:, :], verbosity=1)

# Initialize a list to store the inferred spikes for each trace
inferred_spikes = []

# # For each trace, append the inferred spikes along with other trace information
# for trace in tqdm(traces, desc="Inferring spikes"):
#     inferred_spikes.append({
#         **keyn,  # Include the key information
#         'mask': trace['mask'],  # Include the mask for the trace
#         'fluo_channel': trace['fluo_channel'],  # Include the fluorescence channel information
#         'activity_trace': spikes  # Store the inferred activity trace (spikes)
#     })


In [ ]:
f = np.vstack(dFF)
normalized_f = (f - np.min(f, axis=1, keepdims=True)) / (np.max(f, axis=1, keepdims=True) - np.min(f, axis=1, keepdims=True)) * 2

In [ ]:
from cascade2p.utils import plot_dFF_traces, plot_noise_level_distribution, plot_noise_matched_ground_truth
import matplotlib.pyplot as plt
framerate = (scan.ScanInfo & key).fetch1('fps')

plt.rcParams['figure.figsize'] = [13, 13]

nb_neurons = 20
neuron_indices = np.random.randint(f.shape[0], size=nb_neurons)
# time_axis = plot_dFF_traces(dFF,neuron_indices,framerate, spikes,y_range=(-1.5, 2))
time_axis = plot_dFF_traces(normalized_f,neuron_indices,framerate, spikes, y_range=(-1.5, 2))

plt.title(f"{scan_key['scan_id']} {title} {scan_key['paramset_idx']} {scan_key['curation_id']}")

In [ ]:
key = key & 'extraction_method = "suite2p_deconvolution"'
imaging.Activity.Trace & key

In [ ]:
# Fetch the fluorescence traces from the database, ordered by mask
traces = (imaging.Fluorescence.Trace & key).fetch(as_dict=True, order_by="mask")


In [ ]:
traces

In [ ]:
imaging.Activity.Trace & key

In [ ]:
import tensorflow as tf
print(tf.__version__)

# Compute darksignal for all traces and plot histogram
# Assumes 'darkframes' is available and valid

In [ ]:
darksignals = []
for tr in traces:
    fluo = tr['fluorescence']
    # Use the same darkframes range as for trace0
    if darkframes[-1] > darkframes[0]:
        ds = np.mean(fluo[darkframes[0]:darkframes[-1]])
    else:
        ds = np.mean(fluo[darkframes[0]:darkframes[0]+1])
    darksignals.append(ds)

# Calculate darksignal of the mean of all traces
mean_trace = np.mean([tr['fluorescence'] for tr in traces], axis=0)
if darkframes[-1] > darkframes[0]:
    mean_darksignal = np.mean(mean_trace[darkframes[0]:darkframes[-1]])
else:
    mean_darksignal = np.mean(mean_trace[darkframes[0]:darkframes[0]+1])

plt.figure(figsize=(8, 4))
plt.hist(darksignals, bins=30, color='purple', alpha=0.7)
plt.axvline(mean_darksignal, color='orange', linestyle='--', linewidth=2, label='Mean trace darksignal')
plt.xlabel('Darksignal (mean fluorescence in darkframes)')
plt.ylabel('Count')
plt.title('Histogram of darksignal for all traces')
plt.legend()
plt.show()

In [ ]:

# Calculate darksignal of the mean of all traces efficiently
mean_trace = np.mean([tr['fluorescence'] for tr in traces], axis=0)
mean_darksignal = np.mean(mean_trace[darkframes[0]:darkframes[-1]])
print('Mean trace darksignal:', mean_darksignal)

In [ ]:
mean_darksignal = calculate_mean_darksignal(traces, scan_key)
print(mean_darksignal)

In [ ]:
    shuttertime=0.09
    darkframe_shutter_start = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_start_time')
    darkframe_shutter_end = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_end_time')
    darkframetimes = [darkframe_shutter_end[0] + shuttertime, darkframe_shutter_start[1] - shuttertime]
    twoptimestamps = (event.Event()  &  'event_type LIKE "%2p_frames%"' &  scan_key ).fetch('event_start_time')
    darkframes = sh.get_closest_timestamps(darkframetimes, twoptimestamps)
    darkframes

In [ ]:
# Darkframe 
event.Event & scan_key & 'event_type LIKE "%2p_frames%"' # check if the darkframe event is present in the database

In [ ]:

# from the event table get the main recording gate start / end timestamps.
darkframe_shutter_start = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_start_time')
darkframe_shutter_end = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_end_time')

darkframe_shutter_start = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_start_time')
darkframe_shutter_end = (event.Event()  &  "event_type='shutter'" &  scan_key ).fetch('event_end_time')

shuttertime = 0.09

darkframetimes = [darkframe_shutter_end[0] + shuttertime, darkframe_shutter_start[1] - shuttertime]

# #  and 2p timestamps (which will always be in the recording gate).
twoptimestamps = (event.Event()  &  'event_type LIKE "%2p_frames%"' &  scan_key ).fetch('event_start_time')

darkframes = sh.get_closest_timestamps(darkframetimes, twoptimestamps) #smoothing windwo from above

In [ ]:
darkframes

In [ ]:
imaging.Fluorescence.Trace()


In [ ]:
traces = (imaging.Fluorescence.Trace & scan_key).fetch('fluorescence')
# Stack all traces into a 2D array and then compute the mean trace
traces_stack = np.vstack(traces)
average_trace = np.mean(traces_stack, axis=0)
darksignal = np.mean(average_trace[darkframes[0]:darkframes[-1]])
print('Mean darksignal:', darksignal)

In [ ]:
# traces_stack = np.vstack(traces)
average_trace = np.mean(traces, axis=0)
darksignal = np.mean(average_trace[darkframes[0]:darkframes[-1]])
print('Mean darksignal:', darksignal)

In [ ]:
print('Mean darksignal:', darksignal)

In [ ]:
traces[darkframes[0]:darkframes[-1]]

In [ ]:
import matplotlib.pyplot as plt

# Plot traces[0] from frame 0 to 100, scaled min to max
trace0 = traces[0]['fluorescence'][0:100]
darksignal = np.mean(trace0[darkframes[0]:darkframes[-1]])
scaled_trace0 = (trace0 - trace0.min()) / (trace0.max() - trace0.min())
plt.figure(figsize=(10, 4))
plt.plot(scaled_trace0)

# Add vertical lines at darkframe indices if available
if 'darkframes' in locals() and darkframes is not None:
    for idx in darkframes:
        if 0 <= idx < 100:
            plt.axvline(idx, color='r', linestyle='--', alpha=0.7, label='darkframe range' if idx == darkframes[0] else None)
    if len(darkframes) > 0:
        plt.legend()

# Add horizontal line at darksignal (scaled)
scaled_darksignal = (darksignal - trace0.min()) / (trace0.max() - trace0.min())
plt.axhline(scaled_darksignal, color='g', linestyle=':', alpha=0.8, label='darksignal')
plt.legend()

plt.title('traces[0] (frames 0-100, min-max scaled)')
plt.xlabel('Frame')
plt.ylabel('Scaled Fluorescence')
plt.show()


In [ ]:
darksignals = []
for tr in traces:
    fluo = tr['fluorescence']
    # Use the same darkframes range as for trace0
    if darkframes[-1] > darkframes[0]:
        ds = np.mean(fluo[darkframes[0]:darkframes[-1]])
    else:
        ds = np.mean(fluo[darkframes[0]:darkframes[0]+1])
    darksignals.append(ds)

plt.figure(figsize=(8, 4))
plt.hist(darksignals, bins=30, color='purple', alpha=0.7)
plt.xlabel('Darksignal (mean fluorescence in darkframes)')
plt.ylabel('Count')
plt.title('Histogram of darksignal for all traces')
plt.show()